# Embedding Model Evaluation

#### Compare three embedding models on the Baldwin Boro regulation PDFs

| Model | Type | Cost |
|---|---|---|
| `all-MiniLM-L6-v2` | ChromaDB default (local) | Free |
| `all-mpnet-base-v2` | HuggingFace sentence-transformers (local) | Free |
| `text-embedding-3-small` | OpenAI API | ~$0.02 / 1M tokens |

Each model gets its own ChromaDB collection. The same queries are run against all three and results are printed side-by-side.


In [1]:
import os
import re
import chromadb
import pypdf
import tiktoken
import huggingface_hub
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from chromadb.utils.embedding_functions import (
    DefaultEmbeddingFunction,
    SentenceTransformerEmbeddingFunction,
    OpenAIEmbeddingFunction,
)

load_dotenv(find_dotenv())

try:
    huggingface_hub.login(token=os.environ.get("HUGGINGFACE_API_KEY"), add_to_git_credential=False)
    print("HuggingFace: authenticated")
except Exception as e:
    print(f"HuggingFace: proceeding without auth ({e})")

SECTION_PATTERN = re.compile(
    r'(?=(?:Chapter\s+[A-Z]?\d+|ARTICLE\s+[IVXLC]+|§\s*[A-Z]?\d+[-\w]*\.))',
    re.MULTILINE
)
OVERLAP_CHARS = 200

def extract_sections(pdf_path: Path) -> list[dict]:
    reader = pypdf.PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    raw_chunks = SECTION_PATTERN.split(full_text)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    sections = []
    for i, chunk in enumerate(raw_chunks):
        overlap = raw_chunks[i - 1][-OVERLAP_CHARS:] if i > 0 else ""
        heading = chunk.splitlines()[0].strip()
        sections.append({
            "text": overlap + chunk,
            "source": pdf_path.name,
            "chunk_index": i,
            "heading": heading,
        })
    return sections

HuggingFace: proceeding without auth (Invalid user token.)


In [2]:
data_dir = Path("data")
all_sections = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    all_sections.extend(extract_sections(pdf_path))

print(f"Loaded {len(all_sections)} chunks from {len(list(data_dir.glob('*.pdf')))} PDFs")

Loaded 2061 chunks from 3 PDFs


In [3]:
import openai as openai_client

OPENAI_MAX_TOKENS = 7_500  # safely under the 8191 token limit for text-embedding-3-small
OPENAI_BATCH_SIZE = 500    # stay well under OpenAI's 2048 inputs-per-request limit
_enc = tiktoken.get_encoding("cl100k_base")

def sanitize(text: str, max_tokens: int | None = None) -> str:
    text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
    if max_tokens:
        tokens = _enc.encode(text)
        if len(tokens) > max_tokens:
            text = _enc.decode(tokens[:max_tokens])
    return text

def embed_openai_batched(docs: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    client_oa = openai_client.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embeddings = []
    for i in range(0, len(docs), OPENAI_BATCH_SIZE):
        batch = docs[i:i + OPENAI_BATCH_SIZE]
        response = client_oa.embeddings.create(input=batch, model=model)
        embeddings.extend([r.embedding for r in response.data])
        print(f"  OpenAI: embedded {min(i + OPENAI_BATCH_SIZE, len(docs))}/{len(docs)} docs")
    return embeddings

chroma_client = chromadb.EphemeralClient()
collections = {}

# --- local models ---
for model_name, ef in [
    ("minilm (default)", DefaultEmbeddingFunction()),
    ("mpnet (huggingface)", SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2")),
]:
    docs = [sanitize(s["text"]) for s in all_sections]
    col = chroma_client.get_or_create_collection(
        name=model_name.replace(" ", "_").replace("(", "").replace(")", ""),
        embedding_function=ef,
    )
    col.add(
        documents=docs,
        metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
        ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
    )
    collections[model_name] = col
    print(f"Built collection: {model_name}")

# --- OpenAI model (pre-computed embeddings in controlled batches) ---
openai_docs = [sanitize(s["text"], OPENAI_MAX_TOKENS) for s in all_sections]
print("Building OpenAI embeddings...")
openai_embeddings = embed_openai_batched(openai_docs)
col = chroma_client.get_or_create_collection(
    name="text_embedding_3_small_openai",
    embedding_function=OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small"),
)
col.add(
    documents=openai_docs,
    embeddings=openai_embeddings,
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
collections["text-embedding-3-small (openai)"] = col
print("Built collection: text-embedding-3-small (openai)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Built collection: minilm (default)
Built collection: mpnet (huggingface)
Building OpenAI embeddings...
  OpenAI: embedded 500/2061 docs
  OpenAI: embedded 1000/2061 docs
  OpenAI: embedded 1500/2061 docs
  OpenAI: embedded 2000/2061 docs
  OpenAI: embedded 2061/2061 docs
Built collection: text-embedding-3-small (openai)


In [ ]:
def compare(query: str, n_results: int = 3):
    print(f"\nQUERY: {query}\n{'=' * 80}")
    for model_name, col in collections.items():
        results = col.query(query_texts=[query], n_results=n_results)
        print(f"\n--- {model_name.upper()} ---")
        for text, meta, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            print(f"  Score: {1 - distance:.3f} | {meta['source']} — {meta['heading']}")

compare("do i have to register with the boro if i want to install an alarm?")
compare("what are the rules for parking on residential streets?")
compare("how do i appeal a zoning decision?")
compare("What company is authorized to provide and maintain the telephone service within the boro?")
compare("I'd like to put an alarm system in my business in the borough.  What are the requirements to do this?")
compare("Can I put solar panels on my house roof?  What are the steps for approval?") 
compare("I'm a police officer and would like to work for the borough until I'm 67 years old.  Is this possible?")
compare("I have a construction company, how can I be included on the list of qualified suppliers for borough contracts? ")


QUERY: do i have to register with the boro if i want to install an alarm?

--- MINILM (DEFAULT) ---
  Score: 0.142 | BaldwinBoro-GeneralLegislation.pdf — §  62-6. Duties of alarm user.
  Score: 0.089 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company; permit required; fee.
  Score: 0.079 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company;

--- MPNET (HUGGINGFACE) ---
  Score: 0.615 | BaldwinBoro-GeneralLegislation.pdf — §  62-6. Duties of alarm user.
  Score: 0.582 | BaldwinBoro-GeneralLegislation.pdf — §  62-3. Registration required; application; fees; transferability; false statements.
  Score: 0.581 | BaldwinBoro-GeneralLegislation.pdf — §  62-7. Duties of alarm company;

--- TEXT-EMBEDDING-3-SMALL (OPENAI) ---
  Score: 0.641 | BaldwinBoro-GeneralLegislation.pdf — §  62-3. Registration required; application; fees; transferability; false statements.
  Score: 0.633 | BaldwinBoro-GeneralLegislation.pdf — §  62-6. Duties of alarm user.
  Score: